# 本文件展示了LGflow与torch在链式求导以及梯度更新时的对比

In [1]:
from torch import nn as t_nn
from torch import optim as t_optim
import torch

In [2]:
from LG_flow import nn as l_nn
from LG_flow import optim as l_optim
import LG_flow

# 一. 创建模型

## 1.1 创建torch模型
    创建一个两层全连接的torch模型。

In [3]:
class TNet(t_nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = t_nn.Linear(3, 6)
        self.norm = t_nn.LayerNorm(6)
        self.act1 = t_nn.ReLU()
        self.fc2 = t_nn.Linear(12, 3)
        self.act2 = t_nn.Softmax()
        

    def forward(self, x):                       # [2, 6]
        x = x.reshape(4, 3)                     # [4, 3]
        x = self.fc1(x)                         # [4, 6]
        x = self.norm(x)
        x = self.act1(x)

        xs = x.split(2, 1)                      # [4, 2]* 3
        xs = [s.permute((1, 0)) for s in xs]    # [2, 4]* 3
        x = torch.concat(xs, 1)                 # [2, 12]

        x = self.fc2(x)                         # [2, 3]
        x = self.act2(x)
        return x

t_net = TNet()

In [4]:
t_net

TNet(
  (fc1): Linear(in_features=3, out_features=6, bias=True)
  (norm): LayerNorm((6,), eps=1e-05, elementwise_affine=True)
  (act1): ReLU()
  (fc2): Linear(in_features=12, out_features=3, bias=True)
  (act2): Softmax(dim=None)
)

## 1.2 创建LGflow模型
    通过LGflow，创建一个结构相同的模型。

In [5]:
class LNet(l_nn.Module):
    def __init__(self):
        super().__init__()

        self.fc1 = l_nn.Linear(3, 6)
        self.norm = l_nn.LayerNorm(6)
        self.act1 = l_nn.ReLU()
        self.fc2 = l_nn.Linear(12, 3)
        self.act2 = l_nn.Softmax()

    def forward(self, x):
        x = x.reshape((4, 3))
        x = self.fc1(x)
        x = self.norm(x)
        x = self.act1(x)

        xs = LG_flow.split(x,2, 1)
        xs = [s.permute((1, 0)) for s in xs]
        x = LG_flow.concat(xs, 1)

        x = self.fc2(x)
        x = self.act2(x)
        return x

l_net = LNet()

In [6]:
print(l_net)

## 1.3 使LGflow模型的参数与torch模型参数相同

In [7]:
l_net.fc1.weights = l_nn.Parameter(t_net.fc1.weight.data.numpy())
l_net.fc1.bias = l_nn.Parameter(t_net.fc1.bias.data.numpy())
l_net.fc2.weights = l_nn.Parameter(t_net.fc2.weight.data.numpy())
l_net.fc2.bias = l_nn.Parameter(t_net.fc2.bias.data.numpy())

# 二. 创建输入

In [8]:
t_x = torch.randn(2, 6)    # input for torch
t_t = torch.tensor([[1, 0, 0], [0, 1, 0]], dtype=torch.float32)    # target for torch

In [9]:
t_x

tensor([[ 0.8448, -1.2025,  0.6955,  0.8219,  2.2877, -0.8203],
        [ 0.5996, -1.1577,  0.4340,  0.1607,  0.4756,  1.0505]])

In [10]:
l_x = LG_flow.Tensor(t_x.data.numpy())    # input for LGflow
l_t = LG_flow.Tensor([[1, 0, 0], [0, 1, 0]])    # target for LGflow

In [11]:
print(l_x)

(Tensor shape=(2, 6) dtype=float32 required_grad=False grad_fn=None 
[[ 0.8447781  -1.202457    0.69552404  0.82189095  2.2876768  -0.82028127]
 [ 0.5995768  -1.1577358   0.43402013  0.16067949  0.47561395  1.05046   ]]
)


# 三. 创建损失函数

In [12]:
t_loss_fn = t_nn.CrossEntropyLoss(reduction='sum')

In [13]:
l_loss_fn = l_nn.CrossEntropyLoss(reduction='sum')

# 四. 创建优化器
    这里使用一个较大的学习率，使参数的更新幅度更大

In [14]:
t_optimizer = t_optim.SGD(t_net.parameters(), lr=0.1)

In [15]:
l_optimizer = l_optim.SGD(l_net.parameters(), lr=0.1)

# 五. 反向传播更新模型参数

    以epoch次反向更新为例，展示了LGflow与torch在链式求导与梯度更新方面具有一致性。

In [16]:
epochs = 20

In [17]:
for epoch in range(epochs):
    print('-'*50 + f'epoch: {epoch}' + '-'*50)
    
    # torch forward
    t_y = t_net(t_x)
    print('t_y: ', t_y)

    # LGflow forward
    l_y = l_net(l_x)
    print('l_y: ', l_y)
    print()
    
    # torch loss
    t_loss = t_loss_fn(t_y, t_t)
    print('t_loss: ', t_loss)

    # LGflow loss
    l_loss = l_loss_fn(l_y, l_t)
    print('l_loss: ', l_loss)
    print()

    # torch backward
    t_optimizer.zero_grad()
    t_loss.backward()
    t_optimizer.step()
    
    # LGflow backward
    l_optimizer.zero_grad()
    l_loss.backward()
    l_optimizer.step()
    
    # check parameters
    ## torch parameters
    for param in t_net.parameters():
        print('torch model paramenters 0 value: ')
        print(param)
        print('torch model paramenters 0 grad: ')
        print(param.grad)
        break
        
    print()
    
    ## LGflow parameters
    for name, param in l_net.parameters().items():
        print('LGflow model paramenters 0 value: ')
        print(param)
        print('LGflow model paramenters 0 grad: ')
        print(param.grad)
        break


--------------------------------------------------epoch: 0--------------------------------------------------
t_y:  tensor([[0.3504, 0.2641, 0.3855],
        [0.3753, 0.4101, 0.2147]], grad_fn=<SoftmaxBackward0>)
l_y:  (Tensor shape=(2, 3) dtype=float32 required_grad=False grad_fn=<LG_flow.Math_op.DIV_WITH_TENSOR object at 0x75c87cdc0100> 
[[0.35043436 0.26407924 0.3854864 ]
 [0.37527582 0.41006103 0.21466315]]
)

t_loss:  tensor(2.1082, grad_fn=<NegBackward0>)
l_loss:  (Tensor shape=() dtype=float32 required_grad=False grad_fn=<LG_flow.Math_op.SUM object at 0x75c87cdc05b0> 
2.108241319656372
)

torch model paramenters 0 value: 
Parameter containing:
tensor([[ 0.3340, -0.3437, -0.4130],
        [ 0.5444,  0.2633, -0.2285],
        [ 0.1471,  0.2282,  0.0428],
        [ 0.2885, -0.2972, -0.3876],
        [ 0.5077,  0.4153,  0.4572],
        [-0.4768,  0.4838,  0.0734]], requires_grad=True)
torch model paramenters 0 grad: 
tensor([[-0.0134,  0.0672, -0.0442],
        [-0.0828, -0.4524,  0

/home/lg/anaconda3/envs/lgflow_env/lib/python3.8/site-packages/torch/nn/modules/module.py:1553: UserWarning: Implicit dimension choice for softmax has been deprecated. Change the call to include dim=X as an argument.
  return self._call_impl(*args, **kwargs)


l_y:  (Tensor shape=(2, 3) dtype=float32 required_grad=False grad_fn=<LG_flow.Math_op.DIV_WITH_TENSOR object at 0x75c87cdc0580> 
[[0.9420679  0.02799031 0.02994175]
 [0.03205166 0.9490695  0.01887887]]
)

t_loss:  tensor(1.1738, grad_fn=<NegBackward0>)
l_loss:  (Tensor shape=() dtype=float32 required_grad=False grad_fn=<LG_flow.Math_op.SUM object at 0x75c87cde6040> 
1.173757791519165
)

torch model paramenters 0 value: 
Parameter containing:
tensor([[ 0.4277, -0.3196, -0.3534],
        [ 0.5521,  0.3768, -0.3106],
        [-0.0068,  0.1994,  0.1345],
        [ 0.3230, -0.2609, -0.3713],
        [ 0.4925,  0.3818,  0.4882],
        [-0.4436,  0.3722, -0.0431]], requires_grad=True)
torch model paramenters 0 grad: 
tensor([[-0.0208, -0.0067, -0.0120],
        [ 0.0045, -0.0162,  0.0117],
        [ 0.0188, -0.0101, -0.0093],
        [-0.0041, -0.0174, -0.0054],
        [ 0.0046, -0.0002,  0.0092],
        [-0.0029,  0.0505,  0.0058]])

LGflow model paramenters 0 value: 
(Parameter shape=(6